# Method Resolution Order (MRO) in Python OOP
## Beginner → Intermediate → Advanced

**Learning objectives**

By the end of this notebook, you will be able to:

1. Explain what Method Resolution Order (MRO) means.
2. Trace method lookup in single, multilevel, and multiple inheritance.
3. Use `__mro__` and `.mro()`.
4. Understand the Diamond Problem.
5. Explain Python's C3 Linearization at an intuitive and practical level.
6. Understand how `super()` follows the MRO.
7. Build cooperative multiple-inheritance classes.
8. Debug common MRO and `super()` mistakes.
9. Solve MRO tracing problems from beginner to advanced level.

> **Key idea:** When an object receives a method call such as `obj.show()`, Python needs an ordered list of classes to search. That ordered list is the **Method Resolution Order (MRO)**.


## 1. What is inheritance?

Inheritance allows one class to reuse or extend another class.

- **Parent / base / superclass**: the class being inherited from.
- **Child / derived / subclass**: the class that inherits.

Example:

```text
Animal
   ↑
 Dog
```

`Dog` can use methods defined in `Animal`, unless it overrides them.


In [1]:
class Animal:
    def speak(self):
        print("Animal speaks")

class Dog(Animal):
    pass

dog = Dog()
dog.speak()


Animal speaks


### What happened?

Python did not find `speak()` directly inside `Dog`.

It searched `Dog`, then its parent `Animal`, found `speak()`, and executed it.

For this simple case, the MRO is:

```text
Dog → Animal → object
```

`object` is the ultimate base class of normal Python classes.


## 2. Seeing the MRO

Python provides two common ways to inspect MRO:

```python
ClassName.mro()
ClassName.__mro__
```

`.mro()` returns a list.

`__mro__` returns a tuple.


In [2]:
print(Dog.mro())
print(Dog.__mro__)


[<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>]
(<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


### Reading the result

You should see something equivalent to:

```text
[Dog, Animal, object]
```

This means Python searches in this order:

1. `Dog`
2. `Animal`
3. `object`

The first class containing the requested attribute/method is used.


## 3. Method overriding and MRO

A child class can override a parent's method.

Python searches from the beginning of the MRO, so the child's implementation wins.


In [ ]:
class Animal:
    def speak(self):
        print("Animal speaks")

class Dog(Animal):
    def speak(self):
        print("Dog barks")

dog = Dog()
dog.speak()

print(Dog.mro())


### Important rule

For:

```python
dog.speak()
```

Python effectively asks:

```text
Does Dog have speak?       Yes → use it.
```

It does **not** continue searching after finding the first match.

This "first match in MRO" idea is one of the most important concepts in Python inheritance.


# 4. Multilevel inheritance

Consider:

```text
Grandparent
     ↑
   Parent
     ↑
    Child
```

Example:


In [ ]:
class Grandparent:
    def show(self):
        print("Grandparent")

class Parent(Grandparent):
    pass

class Child(Parent):
    pass

obj = Child()
obj.show()

print(Child.mro())


The MRO is:

```text
Child → Parent → Grandparent → object
```

If `Child` does not contain `show()`, Python checks `Parent`, then `Grandparent`.


# 5. Why do we need MRO?

MRO becomes especially important when a class has **multiple parents**.

Example:

```text
A       B
 \     /
    C
```

`C` inherits from both `A` and `B`.

Now suppose both `A` and `B` define the same method.

Which method should Python use?

MRO gives Python a deterministic answer.


In [ ]:
class A:
    def show(self):
        print("A")

class B:
    def show(self):
        print("B")

class C(A, B):
    pass

obj = C()
obj.show()

print(C.mro())


The MRO is:

```text
C → A → B → object
```

Therefore:

```python
obj.show()
```

finds `show()` in `A` first.

If we reverse the parent order:

```python
class C(B, A):
    pass
```

the MRO changes.


In [ ]:
class C2(B, A):
    pass

obj2 = C2()
obj2.show()

print(C2.mro())


## 6. Parent order matters

Compare:

```python
class C(A, B):
    pass
```

with:

```python
class C(B, A):
    pass
```

The inheritance declaration itself affects the MRO.

This is why the order of base classes is significant in multiple inheritance.


# 7. The Diamond Problem

A classic multiple-inheritance structure is called the **Diamond Problem**.

```text
        A
       / \
      B   C
       \ /
        D
```

Here:

- `B` inherits from `A`
- `C` inherits from `A`
- `D` inherits from both `B` and `C`

Without a consistent ordering rule, Python could potentially encounter `A` more than once or produce contradictory lookup orders.

Python solves this using **C3 Linearization**.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(A):
    pass

class D(B, C):
    pass

print(D.mro())

obj = D()
obj.show()


The MRO is:

```text
D → B → C → A → object
```

Notice that `A` appears only once.

That is a major benefit of Python's MRO algorithm.


# 8. C3 Linearization — the idea

Python uses **C3 Linearization** to construct the MRO for multiple inheritance.

You do not need to memorize the full algorithm immediately. First understand its goals.

A valid MRO should:

1. Preserve the local parent order.
2. Preserve the ordering of ancestors.
3. Avoid duplicating a class.
4. Produce a consistent order.

For:

```python
class D(B, C):
    pass
```

Python must respect:

```text
D before B
D before C
B before A
C before A
```

The resulting order is:

```text
D → B → C → A → object
```


## 9. C3 linearization with a practical merge example

A useful way to think about C3 is as a controlled merge of inheritance sequences.

For:

```text
D(B, C)
```

we have roughly:

```text
L(D) = [D] + merge(L(B), L(C), [B, C])
```

For the diamond:

```text
L(A) = [A, object]
L(B) = [B, A, object]
L(C) = [C, A, object]
```

The merge produces:

```text
[D, B, C, A, object]
```

The full formal algorithm is useful for advanced work, but the practical rule is:

> Python selects a class only when doing so does not violate the ordering constraints of the remaining lists.

The next examples make this easier to see.


# 10. `super()` is not simply "call my parent"

This is a very important misconception.

Many beginners think:

```python
super().method()
```

means:

> "Call my immediate parent."

That is not the most accurate explanation.

A better explanation is:

> `super()` creates a proxy that continues attribute lookup **after the current class in the MRO**.

This distinction becomes crucial in multiple inheritance.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(B):
    def show(self):
        print("C")
        super().show()

obj = C()
obj.show()

print(C.mro())


The output follows:

```text
C
B
A
```

because the MRO is:

```text
C → B → A → object
```

When `C.show()` calls `super().show()`, lookup continues after `C`.

It finds `B.show()`.

Then `B.show()` calls `super().show()`, and lookup continues after `B`.

It finds `A.show()`.


# 11. `super()` with multiple inheritance

Now consider:

```text
      A
     / \
    B   C
     \ /
      D
```

Each class cooperatively calls `super()`.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(A):
    def show(self):
        print("C")
        super().show()

class D(B, C):
    def show(self):
        print("D")
        super().show()

obj = D()
obj.show()

print(D.mro())


Expected order:

```text
D
B
C
A
```

The MRO is:

```text
D → B → C → A → object
```

Notice something surprising:

When `B` calls `super()`, it does **not necessarily call `A` directly**.

The next class after `B` in the MRO is `C`.

So:

```text
B → super() → C
```

This is the key to understanding cooperative multiple inheritance.


# 12. Visualizing the previous example

The execution path is:

```text
D.show()
   ↓
super()
   ↓
B.show()
   ↓
super()
   ↓
C.show()
   ↓
super()
   ↓
A.show()
```

The path is controlled by the MRO, not simply by the parent written next to the current class.


# 13. Cooperative multiple inheritance

A group of classes is **cooperative** when each class participates in the inheritance chain by calling `super()` appropriately.

Example:

```python
class A:
    def process(self):
        print("A")
        super().process()
```

However, this example has a problem because `object` does not normally provide `process()`.

A common advanced technique is to use a suitable endpoint/base class or design the chain so that the final method does not call further.


In [ ]:
class A:
    def process(self):
        print("A")

class B(A):
    def process(self):
        print("B")
        super().process()

class C(A):
    def process(self):
        print("C")
        super().process()

class D(B, C):
    def process(self):
        print("D")
        super().process()

D().process()


Here the chain terminates safely at `A.process()` because `A` does not call `super().process()`.

This pattern is common in real Python frameworks where mixins cooperate through a shared MRO.


# 14. Mixins and MRO

A **mixin** is a small class designed to add a particular capability.

For example:

- logging
- serialization
- validation
- timestamps
- permissions

Mixins often rely heavily on MRO and `super()`.


In [ ]:
class LoggingMixin:
    def save(self):
        print("Logging before save")
        super().save()

class ValidationMixin:
    def save(self):
        print("Validation before save")
        super().save()

class DatabaseModel:
    def save(self):
        print("Saving to database")

class User(LoggingMixin, ValidationMixin, DatabaseModel):
    pass

user = User()
user.save()

print(User.mro())


The MRO is:

```text
User
→ LoggingMixin
→ ValidationMixin
→ DatabaseModel
→ object
```

So the call chain becomes:

```text
Logging
→ Validation
→ Database save
```

This is one of the most useful real-world applications of MRO.


# 15. MRO and constructors

MRO also affects `__init__()`.

With multiple inheritance, cooperative constructors should generally use `super()`.


In [ ]:
class A:
    def __init__(self):
        print("A.__init__")

class B(A):
    def __init__(self):
        print("B.__init__")
        super().__init__()

class C(A):
    def __init__(self):
        print("C.__init__")
        super().__init__()

class D(B, C):
    def __init__(self):
        print("D.__init__")
        super().__init__()

D()


The constructor chain follows the same MRO:

```text
D → B → C → A
```

Expected output:

```text
D.__init__
B.__init__
C.__init__
A.__init__
```

This is why blindly writing `A.__init__(self)` in a multiple-inheritance hierarchy can be problematic: it bypasses the cooperative MRO.


# 16. Passing arguments through `super()`

A common advanced pattern is passing keyword arguments through a cooperative chain.

Each class consumes the arguments it owns and forwards the rest.


In [ ]:
class Base:
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

class NameMixin(Base):
    def __init__(self, name, **kwargs):
        self.name = name
        super().__init__(**kwargs)

class AgeMixin(Base):
    def __init__(self, age, **kwargs):
        self.age = age
        super().__init__(**kwargs)

class Person(NameMixin, AgeMixin):
    pass

p = Person(name="Nitin", age=22)

print(p.name)
print(p.age)
print(Person.mro())


### Why `**kwargs`?

It allows each class in the cooperative chain to consume the arguments it needs while forwarding the remaining arguments.

This technique appears frequently in advanced Python libraries and frameworks.


# 17. Direct parent call vs `super()`

Compare these approaches.

### Direct parent call

```python
Parent.method(self)
```

This explicitly chooses a particular class.

### `super()`

```python
super().method()
```

This follows the MRO.

In simple single inheritance, both can appear to work similarly.

In multiple inheritance, they can behave very differently.


In [ ]:
class A:
    def greet(self):
        print("A")

class B(A):
    def greet(self):
        print("B")
        A.greet(self)

class C(A):
    def greet(self):
        print("C")
        super().greet()

class D(B, C):
    def greet(self):
        print("D")
        super().greet()

D().greet()
print(D.mro())


The direct call:

```python
A.greet(self)
```

jumps directly to `A`.

It does not ask the MRO which class should come next.

That is why direct parent calls can break cooperative multiple inheritance.


# 18. A very important MRO tracing technique

When solving an MRO question, use these steps:

### Step 1
Write the inheritance graph.

### Step 2
Find the class's MRO:

```python
print(ClassName.mro())
```

### Step 3
For each method call, start at the object's class.

### Step 4
Move through the MRO from left to right.

### Step 5
Stop at the first class that defines the requested method.

### Step 6
If that method calls `super()`, continue from the next position in the MRO.

This method makes even complicated examples much easier to trace.


# 19. Example: tracing an advanced hierarchy

Consider:

```text
        A
      /   \
     B     C
      \   / \
       D   E
        \ /
         F
```

Instead of guessing, inspect the MRO.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(A):
    def show(self):
        print("C")
        super().show()

class D(B, C):
    def show(self):
        print("D")
        super().show()

class E(C):
    def show(self):
        print("E")
        super().show()

class F(D, E):
    def show(self):
        print("F")
        super().show()

print(F.mro())

F().show()


### How to solve it

Do not try to memorize the output.

First inspect:

```python
F.mro()
```

Then follow that exact order.

Every `super().show()` moves to the next applicable class in that MRO.

This is the safest method for debugging complex inheritance.


# 20. When Python rejects an inheritance hierarchy

Not every multiple-inheritance combination is possible.

Python raises a `TypeError` when the requested hierarchy cannot produce a consistent MRO.

For example, contradictory base-class ordering can create an impossible hierarchy.

The important lesson is:

> Python prefers to reject an inconsistent MRO rather than silently use an ambiguous lookup order.


In [ ]:
# Uncomment to see Python reject an inconsistent hierarchy.

# class A:
#     pass
#
# class B(A):
#     pass
#
# class C(A):
#     pass
#
# class Impossible(B, C):
#     pass
#
# print(Impossible.mro())


The example above is actually valid, because `B, C` is consistent.

So let's deliberately construct a contradictory ordering.


In [ ]:
class A:
    pass

class B(A):
    pass

class C(A):
    pass

class X(B, C):
    pass

# This attempts to require C before B,
# while X already requires B before C.
# Uncomment to observe the TypeError.

# class Y(C, B, X):
#     pass


The exact error depends on the hierarchy, but the important concept is **MRO inconsistency**.

Python will not create a class when the base-class constraints cannot be satisfied consistently.


# 21. `type(obj).__mro__`

MRO belongs to the class of an object.

For an object:

```python
obj
```

you can inspect:

```python
type(obj).__mro__
```


In [ ]:
class Animal:
    pass

class Dog(Animal):
    pass

dog = Dog()

print(type(dog))
print(type(dog).__mro__)


This is useful when debugging an unfamiliar object.

You can ask:

```python
type(obj).mro()
```

to discover where Python will search for methods.


# 22. MRO and attributes, not only methods

MRO controls lookup for attributes as well.

For example, class variables can be resolved through the MRO.


In [ ]:
class A:
    value = "A"

class B(A):
    pass

class C(A):
    value = "C"

class D(B, C):
    pass

obj = D()

print(obj.value)
print(D.mro())


Python searches:

```text
D → B → C → A → object
```

`D` has no `value`.

`B` has no `value`.

`C` has `value`.

Therefore:

```text
C
```

is returned.


# 23. Attribute lookup: a simplified mental model

For normal instance attribute access such as:

```python
obj.name
```

Python's complete lookup process also considers the instance dictionary, descriptors, and class attributes.

For understanding inheritance, a useful simplified model is:

```text
instance-related lookup
        ↓
class / MRO lookup
        ↓
D → B → C → A → object
```

The real attribute lookup rules are more detailed because Python supports descriptors, properties, `__getattribute__`, and `__getattr__`.

So remember:

> MRO explains class inheritance lookup, but Python attribute lookup has additional machinery.


# 24. `__getattribute__` and advanced lookup

Advanced Python allows customization of attribute access.

Example:


In [ ]:
class Demo:
    def __getattribute__(self, name):
        print(f"Looking for: {name}")
        return object.__getattribute__(self, name)

    value = 100

d = Demo()
print(d.value)


This is beyond basic MRO, but it explains why real attribute lookup can involve more than simply scanning the MRO.

For normal OOP programming, you usually do **not** need to override `__getattribute__`.


# 25. MRO with class methods

MRO also affects class methods.


In [ ]:
class A:
    @classmethod
    def info(cls):
        print("A", cls.__name__)

class B(A):
    pass

class C(B):
    pass

C.info()
print(C.mro())


The method is inherited through the MRO, while `cls` refers to the class through which the method was accessed (`C` here).


# 26. MRO with `@staticmethod`

Static methods also participate in class attribute lookup.


In [ ]:
class A:
    @staticmethod
    def utility():
        print("Utility from A")

class B(A):
    pass

B.utility()
print(B.mro())


Because `B` does not define `utility`, Python finds it in `A`.

The method itself does not receive `self` or `cls`, but finding the attribute still follows inheritance rules.


# 27. `super()` can be used without explicitly naming the parent

Inside a method, modern Python commonly uses:

```python
super().method()
```

instead of:

```python
super(CurrentClass, self).method()
```

The zero-argument form is preferred in normal Python 3 code.


In [ ]:
class Parent:
    def hello(self):
        print("Parent")

class Child(Parent):
    def hello(self):
        print("Child")
        super().hello()

Child().hello()


The zero-argument `super()` works because Python determines the relevant current class and instance context automatically.


# 28. Common mistake: forgetting `super()`

Consider:


In [ ]:
class A:
    def process(self):
        print("A")

class B(A):
    def process(self):
        print("B")
        # super().process() is missing

class C(B):
    def process(self):
        print("C")
        super().process()

C().process()


Output:

```text
C
B
```

`A` is never reached because `B.process()` does not continue the cooperative chain.

In a design that expects every class to participate, forgetting `super()` can silently stop later behavior.


# 29. Common mistake: calling the wrong parent directly

This pattern:

```python
A.method(self)
```

may skip classes that the MRO intended to visit.

Use `super()` when the class is designed for cooperative inheritance.


# 30. Common mistake: assuming `super()` means parent

Remember:

```python
super()
```

means approximately:

> Start attribute lookup after the current class in the MRO.

Therefore, in:

```text
D → B → C → A
```

inside `B`, `super()` can reach `C`, not necessarily `A`.


# 31. Beginner exercise 1 — predict the MRO

Predict the MRO before running the code.

```python
class A:
    pass

class B(A):
    pass

class C(B):
    pass
```

What is:

```python
C.mro()
```


In [ ]:
class A:
    pass

class B(A):
    pass

class C(B):
    pass

print(C.mro())


### Answer

```text
C → B → A → object
```


# 32. Beginner exercise 2 — method lookup

What will this print?

```python
class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(B):
    def show(self):
        print("C")

C().show()
```


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    pass

class C(B):
    def show(self):
        print("C")

C().show()


### Answer

```text
C
```

because `C` is first in the MRO and defines `show()`.


# 33. Intermediate exercise — multiple inheritance

Predict the output:

```python
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(A):
    def show(self):
        print("C")
        super().show()

class D(B, C):
    def show(self):
        print("D")
        super().show()
```

Run it and compare your answer.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(A):
    def show(self):
        print("C")
        super().show()

class D(B, C):
    def show(self):
        print("D")
        super().show()

D().show()
print("MRO:", D.mro())


### Answer

The output is:

```text
D
B
C
A
```

because:

```text
D → B → C → A → object
```

and every class calls `super()`.


# 34. Intermediate exercise — change parent order

Now change:

```python
class D(B, C):
```

to:

```python
class D(C, B):
```

Predict what happens.


In [ ]:
class D2(C, B):
    def show(self):
        print("D2")
        super().show()

print(D2.mro())
D2().show()


The parent order changes the MRO.

Therefore, parent ordering is not cosmetic. It can change program behavior.


# 35. Advanced exercise — constructor chain

Create a hierarchy where:

```text
A
↑
B   C
 \ /
  D
```

Each `__init__()` should print its class name and call `super()`.

Expected order:

```text
D
B
C
A
```

Try writing it yourself before looking at the solution.


In [ ]:
class A:
    def __init__(self):
        print("A")

class B(A):
    def __init__(self):
        print("B")
        super().__init__()

class C(A):
    def __init__(self):
        print("C")
        super().__init__()

class D(B, C):
    def __init__(self):
        print("D")
        super().__init__()

D()


# 36. Advanced exercise — mixin pipeline

Build a class with three mixins:

```text
LoggingMixin
ValidationMixin
SaveMixin
```

Each should define `save()` and call `super().save()`.

The final class should execute all three operations exactly once.

This is a practical use of MRO.


In [ ]:
class LoggingMixin:
    def save(self):
        print("Logging")
        super().save()

class ValidationMixin:
    def save(self):
        print("Validation")
        super().save()

class SaveMixin:
    def save(self):
        print("Saving")

class User(LoggingMixin, ValidationMixin, SaveMixin):
    pass

user = User()
user.save()

print(User.mro())


# 37. Advanced challenge — explain the output

Study this carefully:

```python
class A:
    def action(self):
        print("A")

class B(A):
    def action(self):
        print("B")
        super().action()

class C(A):
    def action(self):
        print("C")
        super().action()

class D(B, C):
    pass
```

Questions:

1. What is `D.mro()`?
2. What does `D().action()` print?
3. Why does `B.super()` reach `C`?
4. Why is `A` called only once?


In [ ]:
class A:
    def action(self):
        print("A")

class B(A):
    def action(self):
        print("B")
        super().action()

class C(A):
    def action(self):
        print("C")
        super().action()

class D(B, C):
    pass

print("MRO:", D.mro())
print("Output:")
D().action()


### Explanation

1. MRO:

```text
D → B → C → A → object
```

2. `D()` inherits `action()` from `B`, so `B.action()` starts.
3. `B` calls `super()`, which continues after `B` in the MRO.
4. The next class is `C`.
5. `C` calls `super()`, reaching `A`.
6. `A` finishes the chain.

Therefore:

```text
B
C
A
```


# 38. A compact MRO cheat sheet

| Concept | Meaning |
|---|---|
| MRO | Order in which Python searches classes |
| `.mro()` | Returns MRO as a list |
| `.__mro__` | Returns MRO as a tuple |
| `super()` | Continues lookup through the MRO |
| Multiple inheritance | A class has multiple direct bases |
| Diamond problem | Shared ancestor in multiple inheritance |
| C3 Linearization | Algorithm Python uses to construct consistent MRO |
| Cooperative inheritance | Classes use `super()` to participate in a shared chain |
| First match | Python uses the first applicable definition found in the MRO |

### Most important rule

```text
Method lookup → follow the MRO from left to right → use the first match.
```

### Most important `super()` rule

```text
super() → continue lookup after the current class in the MRO.
```


# 39. Final practice set

Try these without running them first.

### Question 1
What is the MRO?

```python
class A: pass
class B(A): pass
class C(B): pass
```

### Question 2
What does `super()` mean in practical terms?

### Question 3
Why is:

```python
super().method()
```

different from:

```python
Parent.method(self)
```

in multiple inheritance?

### Question 4
What is the Diamond Problem?

### Question 5
What algorithm does Python use to construct MRO?

### Question 6
Why should cooperative mixins generally use `super()`?

### Question 7
For:

```text
D → B → C → A → object
```

what class does `super()` inside `B` continue toward?

### Question 8
Why can Python raise `TypeError` when creating a class with multiple inheritance?

### Question 9
What is the difference between:

```python
Class.mro()
```

and:

```python
Class.__mro__
```

### Question 10
How can you debug an unfamiliar object's inheritance lookup?


# 40. Final answers

1. `C → B → A → object`
2. `super()` continues attribute lookup after the current class in the MRO.
3. A direct parent call explicitly selects a class; `super()` follows the cooperative MRO.
4. It is the multiple-inheritance shape where two branches share an ancestor.
5. **C3 Linearization**.
6. So each class can participate in the shared method-resolution chain without hard-coding a specific parent.
7. Toward `C`.
8. Because the requested inheritance relationships can be inconsistent and no valid MRO can satisfy all ordering constraints.
9. `.mro()` returns a list; `.__mro__` is the class's MRO tuple.
10. Use `type(obj).mro()` or `type(obj).__mro__`, then trace the lookup left-to-right.


# 41. Takeaway

If you remember only five things, remember these:

### 1. MRO is an ordered search path

```text
Child → Parent → Grandparent → object
```

### 2. Multiple inheritance creates a more complicated search path

```text
D → B → C → A → object
```

### 3. Python uses C3 Linearization

It creates a consistent MRO while respecting inheritance constraints.

### 4. `super()` follows the MRO

It does **not** simply mean "my parent".

### 5. Cooperative inheritance depends on `super()`

When multiple classes are designed to work together, each class should generally call `super()` so the entire MRO chain can execute.

---

## End of Notebook

**Suggested next step:** Modify the examples yourself—add another mixin, reverse base-class order, remove a `super()` call, and inspect how the MRO and output change. That is one of the fastest ways to become comfortable with Python multiple inheritance.
